# Day 4: S2 и кривая Гильберта (S2 vs H3 на одном полигоне)

**Теория перед ноутбуком:** 3Blue1Brown о кривой Гильберта · [S2 cell hierarchy](https://s2geometry.io/devguide/s2cell_hierarchy.html) · [Perone: S2 на сфере](https://blog.christianperone.com/2015/08/googles-s2-geometry-on-the-sphere-cells-and-hilbert-curve/)

**Что выясняем:** где S2 выигрывает у H3 и наоборот.

| | S2 | H3 |
|---|---|---|
| форма ячейки | квадрат (проекция куба) | гексагон (+12 пентагонов) |
| площадь внутри уровня | **неравная** (разброс ~2x) | почти равная |
| иерархия | **строгая**: родитель = префикс id | приблизительная, дети не складываются в родителя |
| порядок id | **кривая Гильберта** → соседство по id ≈ соседство на карте | порядка нет |
| соседи | 4 ребра (+ углы), расстояния неравные | 6 соседей, все равноудалённые |

**Гипотеза, которую проверяем числами:** покрытие S2 раскладывается в небольшое число *непрерывных диапазонов id* (range scan по B-tree), а H3 — только в список ключей.

**Что остаётся после сессии:** таблица сравнения, карта `m01/s2_vs_h3.html`, строки в `numbers.md`, 3–4 строки в `stories.md`.

In [ ]:
import math, os, json
import numpy as np, pandas as pd
import h3, s2sphere as s2
from shapely.geometry import Polygon, mapping, shape
from shapely.ops import unary_union
from pyproj import Geod

OFFICE = (43.2380, 76.9450)        # Алматы, тот же ориентир, что в m1_d3
R_KM = 6371.0088                   # средний радиус Земли, для площади ячеек S2
GEOD = Geod(ellps="WGS84")
os.makedirs("m01", exist_ok=True)
print(h3.__version__)

## Шаг 1 — один полигон на всё сравнение

Берём реальную административную границу (район Алматы) через OSMnx и кешируем её в
`data/processed/`, чтобы ноутбук не зависел от сети. Если геокодер недоступен — честный fallback:
квадрат ~10x10 км вокруг офиса (в выводах пометь, какой вариант использовал).

**Ловушка геокодера:** на `"Bostandyk District, Almaty, Kazakhstan"` Nominatim отдаёт здание
на 0.0002 км2, а на `"Almaty, Kazakhstan"` — Алматинскую **область** (104 806 км2). Отсюда
проверка площади на разумный диапазон: молча взятый не тот полигон испортит все числа ниже.

In [ ]:
POLY_CACHE = "../../data/processed/m01_polygon.geojson"
PLACE = "Бостандыкский район, Алматы, Казахстан"   # англ. "Bostandyk District" Nominatim отдаёт как здание!
SANE_KM2 = (10, 400)                               # а "Almaty, Kazakhstan" — это область на 104 806 км2

def load_polygon():
    if os.path.exists(POLY_CACHE):
        with open(POLY_CACHE, encoding="utf-8") as f:
            gj = json.load(f)
        return shape(gj["geometry"]), gj["properties"]["source"]
    try:
        import osmnx as ox
        gdf = ox.geocode_to_gdf(PLACE)
        poly = gdf.geometry.iloc[0]
        km2 = abs(GEOD.geometry_area_perimeter(poly)[0]) / 1e6
        if not (SANE_KM2[0] <= km2 <= SANE_KM2[1]):   # геокодер попал не туда
            raise ValueError(f"{PLACE}: {km2:.2f} км2 вне {SANE_KM2}")
        src = f"OSM: {PLACE}"
    except Exception as e:                      # нет сети / геокодер молчит / не тот объект
        print("geocode failed:", type(e).__name__, e)
        dlat, dlon = 0.045, 0.062               # ~10 x 10 км
        poly = Polygon([(OFFICE[1]-dlon, OFFICE[0]-dlat), (OFFICE[1]+dlon, OFFICE[0]-dlat),
                        (OFFICE[1]+dlon, OFFICE[0]+dlat), (OFFICE[1]-dlon, OFFICE[0]+dlat)])
        src = "fallback square ~10x10 km around OFFICE"
    os.makedirs(os.path.dirname(POLY_CACHE), exist_ok=True)
    with open(POLY_CACHE, "w", encoding="utf-8") as f:
        json.dump({"type": "Feature", "properties": {"source": src},
                   "geometry": mapping(poly)}, f)
    return poly, src

POLY, POLY_SRC = load_polygon()
POLY_KM2 = abs(GEOD.geometry_area_perimeter(POLY)[0]) / 1e6
print(POLY_SRC)
print(f"площадь полигона: {POLY_KM2:.2f} км2, вершин: {len(POLY.exterior.coords)}")

## Шаг 2 — покрытие H3

Два режима, разница важна для ответа «почему ring search нужен финальный re-rank»:

* `h3shape_to_cells` — ячейка попадает, если **её центр** внутри полигона (край может остаться непокрытым);
* `h3shape_to_cells_experimental(..., contain="overlap")` — любое пересечение (покрытие с запасом).

In [ ]:
def cover_h3(poly, res, mode="center"):
    shp = h3.geo_to_h3shape(mapping(poly))
    if mode == "center":
        return list(h3.h3shape_to_cells(shp, res))
    return list(h3.h3shape_to_cells_experimental(shp, res, contain="overlap"))

h3_cells = {(res, mode): cover_h3(POLY, res, mode)
            for res in (7, 8, 9) for mode in ("center", "overlap")}

for k, v in h3_cells.items():
    print(k, len(v))

assert len(h3_cells[(8, "overlap")]) >= len(h3_cells[(8, "center")])
assert len(h3_cells[(9, "center")]) > len(h3_cells[(8, "center")]) > len(h3_cells[(7, "center")])

## Шаг 3 — покрытие S2 (и его ловушка)

**В `s2sphere` нет `S2Polygon`/`S2Loop`** — у `RegionCoverer` из регионов есть только `Cap` и
`LatLngRect`. Поэтому покрытие произвольного полигона делаем в два приёма:

1. `RegionCoverer.get_covering(LatLngRect)` по bbox полигона — отдаёт **разноуровневое** покрытие
   (мелкие ячейки у края, крупные внутри), ограниченное `max_cells`;
2. отбрасываем ячейки, не пересекающие сам полигон (shapely).

`max_cells` — это не «сколько хочу», а бюджет: coverer сам решает, где дробить. Крути
`min_level`/`max_level`/`max_cells` и смотри, как меняется over-coverage.

**Читай `coverage` у S2 с поправкой:** мы покрываем bbox и потом отбрасываем ячейки, не
задевающие полигон, поэтому ~2x запаса — это ограничение `s2sphere`, а **не** свойство S2.
Настоящий `S2RegionCoverer` по `S2Polygon` (C++/Go, или `s2geometry` через pybind) дробил бы
край до `max_level` и дал бы покрытие, сопоставимое с H3 overlap. В выводах это надо назвать
честно, иначе получится подтасовка в пользу H3.

In [ ]:
def s2_cell_polygon(cid: s2.CellId) -> Polygon:
    c = s2.Cell(cid)
    pts = []
    for i in range(4):
        ll = s2.LatLng.from_point(c.get_vertex(i))
        pts.append((ll.lng().degrees, ll.lat().degrees))
    return Polygon(pts)

def s2_cell_area_km2(cid: s2.CellId) -> float:
    return s2.Cell(cid).exact_area() * R_KM * R_KM      # стерадианы -> км2

def cover_s2(poly, min_level, max_level, max_cells, clip=True):
    minx, miny, maxx, maxy = poly.bounds
    rect = s2.LatLngRect(s2.LatLng.from_degrees(miny, minx), s2.LatLng.from_degrees(maxy, maxx))
    rc = s2.RegionCoverer()
    rc.min_level, rc.max_level, rc.max_cells = min_level, max_level, max_cells
    ids = list(rc.get_covering(rect))
    if clip:
        ids = [i for i in ids if s2_cell_polygon(i).intersects(poly)]
    return ids

s2_cells = {("s2", mc): cover_s2(POLY, 10, 14, mc) for mc in (100, 500, 2000)}
for (_, mc), ids in s2_cells.items():
    print("max_cells", mc, "->", len(ids), "ячеек, уровни", sorted({i.level() for i in ids}))

# уровень S2 ~ res H3: L13 ≈ res 8 по площади — проверь по таблице ниже
assert all(10 <= i.level() <= 14 for ids in s2_cells.values() for i in ids)

## Шаг 4 — таблица сравнения

Метрики (считаются ниже, выводы пиши словами):

* `n_cells` — сколько ключей нужно, чтобы описать район;
* `coverage` = площадь объединения ячеек / площадь полигона (>1 — ловим лишнее, <1 — теряем край);
* `inside_frac` — какую долю самого полигона реально накрыли;
* `area_min/med/max`, `area_spread` — разброс площади ячейки внутри покрытия (вот где S2 неравномерен);
* `levels` — сколько разных уровней/разрешений в одном покрытии.

In [ ]:
def stats(name, geoms, areas_km2, levels):
    merged = unary_union(geoms)
    cov_km2 = abs(GEOD.geometry_area_perimeter(merged)[0]) / 1e6
    inter = abs(GEOD.geometry_area_perimeter(merged.intersection(POLY))[0]) / 1e6
    a = np.array(areas_km2)
    return {"scheme": name, "n_cells": len(geoms),
            "coverage": cov_km2 / POLY_KM2,              # >1 = запас, <1 = недобор
            "inside_frac": inter / POLY_KM2,
            "area_min": a.min(), "area_med": float(np.median(a)), "area_max": a.max(),
            "area_spread": a.max() / a.min(), "levels": len(set(levels))}

rows = []
for (res, mode), cells in h3_cells.items():
    rows.append(stats(f"H3 res{res} ({mode})",
                      [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells],
                      [h3.cell_area(c, unit="km^2") for c in cells],
                      [res] * len(cells)))
for (_, mc), ids in s2_cells.items():
    rows.append(stats(f"S2 L10-14 (max_cells={mc})",
                      [s2_cell_polygon(i) for i in ids],
                      [s2_cell_area_km2(i) for i in ids],
                      [i.level() for i in ids]))

cmp = pd.DataFrame(rows).set_index("scheme").round(3)
cmp

In [ ]:
# Проверки к таблице (подправь порог, если взял другой полигон):
# 1) у H3 внутри одного res разброс площадей почти нулевой
assert cmp.loc["H3 res8 (center)", "area_spread"] < 1.1
# 2) у S2 в разноуровневом покрытии разброс кратный
assert cmp.loc["S2 L10-14 (max_cells=500)", "area_spread"] > 2
# 3) режим center недобирает край, overlap — перебирает
assert cmp.loc["H3 res8 (center)", "inside_frac"] < 1.0 <= cmp.loc["H3 res8 (overlap)", "coverage"]
print("ok")

## Шаг 5 — главный аргумент S2: локальность по кривой Гильберта

`CellId` — это int64, позиция на кривой Гильберта; `range_min()/range_max()` дают интервал
листовых id, покрытых ячейкой. Если покрытие укладывается в K непрерывных интервалов, то
«всё в этом районе» = **K range-запросов `WHERE s2_id BETWEEN a AND b`** по обычному B-tree.

Деталь, на которой легко сломаться: у листовых ячеек младший бит — маркер, поэтому соседние
листовые id различаются на **2**, а не на 1 (отсюда `step=2`).

In [ ]:
def hilbert_runs(cell_ids, step=2):
    iv = sorted((c.range_min().id(), c.range_max().id()) for c in cell_ids)
    runs, cur_end = 1, iv[0][1]
    for lo, hi in iv[1:]:
        if lo > cur_end + step:
            runs += 1
        cur_end = max(cur_end, hi)
    return runs

ids = s2_cells[("s2", 500)]
union = s2.CellUnion(list(ids)); union.normalize()
runs = hilbert_runs(list(union.cell_ids()))
print(f"S2: {len(ids)} ячеек -> {union.num_cells()} после normalize -> {runs} диапазонов id")
print("H3: ключей", len(h3_cells[(8, 'center')]), "- диапазонов нет, порядок id не географический")
assert runs < union.num_cells()

**Задание 1.** Докажи, что у H3 географического порядка нет: отсортируй `h3_cells[(8,"center")]`
по `int(cell, 16)` и посчитай среднее `h3.grid_distance` между подряд идущими id. Сравни с тем же
для S2 (подряд идущие `CellId` → расстояние между центрами в метрах, `GEOD.inv`). Одна цифра в вывод.

**Задание 2.** Строгая иерархия S2: возьми ячейку L13, её `parent(10)`, проверь
`parent.contains(child)` и что сумма площадей всех детей L13 внутри родителя равна площади
родителя. Повтори для H3 (`cell_to_parent` / `cell_to_children`) — и объясни в двух строках,
почему у H3 так не выходит и чем это грозит при агрегации спроса по уровням.

**Задание 3.** Убери bbox-фору: напиши свой coverer по полигону — начни с ячеек L10,
каждую, которая пересекает границу, дроби через `cell_id.children()` до `max_level`, внутренние
оставляй как есть. Сравни `coverage` с таблицей выше: насколько `s2sphere` переплачивает за
отсутствие `S2Polygon`?

**Задание 4 (на выбор).** Geofencing: сколько ячеек нужно, чтобы описать зону обслуживания с
точностью ±5% по площади, у S2 и у H3? Это и есть ответ на self-check
«Pick H3, S2 or geohash for (a) surge heatmap, (b) geofencing, (c) nearest driver».

In [ ]:
# Задание 1

In [ ]:
# Задание 2

## Шаг 6 — карта

In [ ]:
import folium
center = (POLY.centroid.y, POLY.centroid.x)
m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")   # cartodbpositron теперь требует API-ключ
folium.GeoJson(mapping(POLY), name="полигон",
               style_function=lambda _: {"color": "#111111", "weight": 2, "fill": False}).add_to(m)

fg = folium.FeatureGroup(name="H3 res 8 (center)", show=True)
for c in h3_cells[(8, "center")]:
    folium.Polygon(h3.cell_to_boundary(c), color="#1F5E96", weight=1,
                   fill=True, fill_opacity=0.12, tooltip=f"H3 {c}").add_to(fg)
fg.add_to(m)

fg2 = folium.FeatureGroup(name="S2 L10-14 (max_cells=500)", show=True)
for i in s2_cells[("s2", 500)]:
    ring = [(y, x) for x, y in s2_cell_polygon(i).exterior.coords]
    folium.Polygon(ring, color="#C4561D", weight=1, fill=True, fill_opacity=0.10,
                   tooltip=f"S2 L{i.level()} {i.to_token()}").add_to(fg2)
fg2.add_to(m)

folium.LayerControl().add_to(m)
m.save("m01/s2_vs_h3.html"); m

## Выводы (заполнить)

* **S2 vs H3 в одной строке:** ...
* Когда беру S2: ...
* Когда беру H3: ...
* Когда хватает geohash: ...

**Перенести:**
- [ ] строки в `numbers.md`: n_cells / coverage / area_spread для H3 res8 и S2 max_cells=500 по тому же полигону + число диапазонов Гильберта (дата, git hash, источник полигона);
- [ ] 3–4 строки в `stories.md`: ответ на «почему Uber ушёл с S2 на H3, а Google/Foursquare остались»;
- [ ] галочка `m1-4` (и `m1-2`, если теория по H3 закрыта) в плане.